In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

CSV_FILENAME = "../datasets/titanic.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (891, 12)


In [2]:
# ==========================================
# 1. ÖZEL FEATURE ENGINEER SINIFI (Pipeline Uyumlu)
# ==========================================


class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):

  def __init__(self):
    self.age_median_ = None
    self.fare_median_ = None
    self.feature_names_in_ = (
    None)  # <-- Backend'in okuyabilmesi için ekliyoruz

  def fit(self, X, y=None):
    # Veri sızıntısını önlemek için medyanlar sadece eğitim setinden öğrenilir
    self.age_median_ = X["Age"].median()
    self.fare_median_ = X["Fare"].median()
    self.feature_names_in_ = X.columns.tolist()
    return self

  def transform(self, X):
    df = X.copy()

    # Title Extraction
    if "Name" in df.columns:
      df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
      df["Title"] = df["Title"].replace(
          [
              "Lady",
              "Countess",
              "Capt",
              "Col",
              "Don",
              "Dr",
              "Major",
              "Rev",
              "Sir",
              "Jonkheer",
              "Dona",
          ],
          "Rare",
      )
      df["Title"] = df["Title"].replace(
          {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
      )

    # Family Size & IsAlone
    if "SibSp" in df.columns and "Parch" in df.columns:
      df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
      df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Deck Extraction
    if "Cabin" in df.columns:
      df["Deck"] = df["Cabin"].fillna("U").astype(str).str[0]
      df["Deck"] = df["Deck"].replace("T", "U")

    # Fitted medyanlar ile eksik verileri doldurma
    if "Age" in df.columns:
      df["Age"] = df["Age"].fillna(self.age_median_)
    if "Fare" in df.columns:
      df["Fare"] = df["Fare"].fillna(self.fare_median_)

    required_features = [
        "Pclass",
        "Sex",
        "Age",
        "Fare",
        "FamilySize",
        "IsAlone",
        "Title",
        "Deck",
    ]
    return df[required_features]


# Ham veriyi alıyoruz (Feature engineering kodları artık pipeline içinde çalışacak)
raw_features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "Name",
    "Cabin",
]
X = df[raw_features].copy()
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
# ==========================================
# 2. PIPELINE KURULUMU
# ==========================================
numeric_cols = ["Age", "Fare", "FamilySize", "IsAlone", "Pclass"]
categorical_cols = ["Sex", "Title", "Deck"]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_cols,
        ),
    ],
    remainder="passthrough",
)

pipeline_model = Pipeline(
    steps=[
        ("feature_engineer", TitanicFeatureEngineer()),
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=30)),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 0.8380
Fold 2: 0.8371
Fold 3: 0.7697
Fold 4: 0.8315
Fold 5: 0.8539

Gerçek K-Fold Başarısı (Ortalama): 0.8260
Skor Sapması (Standart Sapma)    : 0.0292



In [4]:
# ==========================================
# 3. NİHAİ MODEL EĞİTİMİ VE TEST
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))

Train Doğruluk Oranı: 0.8385
Model Doğruluk Oranı (Accuracy): 0.8156

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       105
           1       0.81      0.73      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



In [5]:
# ==========================================
# 4. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/titanic_pipeline.pkl")

print(
    "Uçtan uca Titanic Pipeline modeli başarıyla '../backend/models/'"
    " klasörüne kaydedildi!"
)

Uçtan uca Titanic Pipeline modeli başarıyla '../backend/models/' klasörüne kaydedildi!
